In [25]:
import pandas as pd
from pypdf import PdfReader
import re

In [8]:
pip install pypdf

Note: you may need to restart the kernel to use updated packages.


In [15]:
len(PdfReader('WYCIAG_BANKOWY_8324_378RPG_000_M_0016-2025_WZE_2025-11-29.pdf').pages)#.pages[nr_page].extract_text()
#pd.read_csv()

3

In [18]:
page_content = PdfReader('WYCIAG_BANKOWY_8324_378RPG_000_M_0016-2025_WZE_2025-11-29.pdf').pages[0].extract_text()

In [22]:
beg_phrase = page_content.index('Saldo po operacji')
#end_phrase = page_content.index('Niniejszy wyciąg bankowy jest zestawieniem wszystkich operacji dokonanych na rachunku bankowym')
#        else:
#          beg_phrase = page_content.index('Saldo po operacji')
end_phrase = page_content.index('BNP Paribas Bank Polska Spółka Akcyjna z siedzibą w Warszawie ')
          
       

In [23]:
# a = 
page_content[beg_phrase:end_phrase].split('\n')

['Saldo po operacji',
 '01.11.2025',
 'SALDO POCZĄTKOWE 13,17',
 '03.11.2025',
 'PRZELEW OTRZYMANY ELIXIR',
 'DANIEL  LILLA',
 'UL. BEMA 26  M.4',
 '33-100 TARNÓW',
 '31102049550000750202263465',
 'YOLO',
 'CEN2511031646315',
 '1 500,00 1 513,17',
 '04.11.2025',
 'TRANS.BEZGOT. KARTĄ DEBET.',
 '516842------5081',
 'DANIEL LILLA         Prague',
 'HBO                       CZE',
 '29,99             PLN 2025-11-03',
 'PSD3299895445467',
 '-29,99 1 483,18',
 '04.11.2025',
 'TRANS.BEZGOT. KARTĄ DEBET.',
 '516842------5081',
 'DANIEL LILLA         AMSTERDAM',
 'NETFLIX.COM               NLD',
 '33.00             PLN 2025-11-03',
 'PSD3369543458768',
 '-33,00 1 450,18',
 'Rachunek oszczędnościowo-rozliczeniowy',
 'NRB: 75 1600 1462 1891 2670 5023 3380',
 'IBAN: PL75 1600 1462 1891 2670 5023 3380 - ciąg dalszy',
 'Data księgowania',
 'Rodzaj transakcji',
 'Szczegóły operacji',
 'Obroty na rachunku Saldo po operacji',
 '04.11.2025',
 'TRANS.BEZGOT. KARTĄ DEBET.',
 '516842------5081',
 'DANIEL 

In [24]:
file_ = 'WYCIAG_BANKOWY_8324_378RPG_000_M_0016-2025_WZE_2025-11-29.pdf'
path_to_user_loaded_files = 'WYCIAG_BANKOWY_8324_378RPG_000_M_0016-2025_WZE_2025-11-29.pdf'

In [202]:
df_main = pd.DataFrame()
df_bnp = pd.DataFrame()
if 'WYCIAG_BANKOWY_' in file_:
      for nr_page in range(0,(len(PdfReader(file_).pages))):
        page_content = PdfReader(file_).pages[nr_page].extract_text()
        if nr_page == 0:
          beg_phrase = page_content.index('SALDO POCZĄTKOWE')
          end_phrase = page_content.index('BNP Paribas Bank Polska Spółka Akcyjna z siedzibą w Warszawie ')
        elif nr_page == len(PdfReader(file_).pages)-1:
          if 'Saldo po operacji' not in page_content:
            continue
          else:
            beg_phrase = page_content.index('Saldo po operacji')
            end_phrase = page_content.index('Niniejszy wyciąg bankowy jest zestawieniem wszystkich operacji dokonanych na rachunku bankowym')
        else:
          beg_phrase = page_content.index('Saldo po operacji')
          end_phrase = page_content.index('BNP Paribas Bank Polska Spółka Akcyjna z siedzibą w Warszawie ')
          
        a = page_content[beg_phrase:end_phrase].split('\n')
        
        occurencies = []
        patter = re.compile(r'[A-Z][A-Z][A-Z][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9][0-9]')
        #patter = re.compile(r'[0-3][0-9].[0-1][0-9].[2][0][0-3][0-9]')

        for index,line in enumerate(a):
          if patter.match(line):
            occurencies.append(index)

        for index,i in enumerate(occurencies):
          if index == 0:
            if 'Data waluty' in a[i+1]:
              end_point = i+3
              one_item = a[1:end_point-2]+ [a[end_point-1]]
              if len(one_item) <= 8:
                df_main = pd.concat([df_main, pd.DataFrame(one_item).T])
              else:
                df_main = pd.concat([df_main, pd.DataFrame(one_item[:3] + one_item[4:]).T])
              last_point = end_point
            
            elif 'BLIK' in a[2]:
              end_point = i+2
              one_item = a[1:3] + a[6:end_point]

              if len(one_item) <= 8:
                df_main = pd.concat([df_main, pd.DataFrame(one_item).T])
              else:
                df_main = pd.concat([df_main, pd.DataFrame(one_item[:3] + one_item[4:]).T])
              last_point = end_point

            else:
              end_point = i+2
              one_item = a[1:end_point]
                    
              if len(one_item) <= 8:
                df_main = pd.concat([df_main, pd.DataFrame(one_item).T])
              else:
                df_main = pd.concat([df_main, pd.DataFrame(one_item[:3] + one_item[4:]).T])
              last_point = end_point
          
          # not first page
          else:
            if 'Rachunek oszczędnościowo-rozliczeniowy' in a[last_point]:
              last_point = last_point + 7

            if 'Data waluty' in a[i+1]:
              end_point = i+3
              df_main = pd.concat([df_main, pd.DataFrame(a[last_point:end_point-2]+ [a[end_point-1]]).T])
              last_point = end_point
            elif 'BLIK' in a[last_point+1]:
              end_point = i+2
              one_item = a[last_point:last_point+2] + a[last_point+6:end_point]
              if len(one_item) <= 8:
                df_main = pd.concat([df_main, pd.DataFrame(one_item).T])
              else:
                df_main = pd.concat([df_main, pd.DataFrame(one_item[:3] + one_item[4:]).T])
              last_point = end_point
            elif 'OPŁATA ZA RACHUNEK/PAKIET' in a[last_point+2] or 'OPŽATA ZA RACHUNEK/PAKIET' in a[last_point+2]:
              print(index)
              end_point = i
              df_main = pd.concat([df_main, pd.DataFrame(a[last_point:last_point+5] + ['0','0','0','0']).T])
              one_item = a[last_point+5:end_point]
              if len(one_item) <= 8:
                df_main = pd.concat([df_main, pd.DataFrame(one_item).T])
              else:
                df_main = pd.concat([df_main, pd.DataFrame(one_item[:3] + one_item[4:]).T])
              last_point = end_point +2
            else:
              end_point = i+2
              one_item = a[last_point:end_point]
              if len(one_item) <= 8:
                df_main = pd.concat([df_main, pd.DataFrame(one_item).T])
              else:
                df_main = pd.concat([df_main, pd.DataFrame(one_item[:3] + one_item[4:]).T])
              last_point = end_point
          df_bnp =  pd.concat([df_bnp, df_main])

7


In [266]:
from datetime import datetime
from datetime import date

In [287]:
endDate

datetime.date(2025, 5, 27)

In [289]:
startDate = date.today()
# reconstruct date fully
endDate = date(startDate.year -1, startDate.month, startDate.day)
print(endDate)

2025-05-27


In [299]:
endDate_ = date(startDate.year -1, startDate.month, startDate.day)

In [297]:
df_bnp[0] = pd.to_datetime(df_bnp[0], format="%Y-%m-%d")

In [294]:
df_bnp[0]

0   2025-11-03
0   2025-11-03
0   2025-11-04
0   2025-11-03
0   2025-11-04
       ...    
0   2025-11-25
0   2025-11-26
0   2025-11-26
0   2025-11-26
0   2025-11-28
Name: 0, Length: 669, dtype: datetime64[ns]

In [302]:
df_bnp[0].dt.date < endDate

0    False
0    False
0    False
0    False
0    False
     ...  
0    False
0    False
0    False
0    False
0    False
Name: 0, Length: 669, dtype: bool

In [245]:
lol = df_bnp[0].drop_duplicates().to_list()#.append('All')

In [246]:
lol.append('All')

In [247]:
lol

['03.11.2025',
 '04.11.2025',
 '05.11.2025',
 '06.11.2025',
 '07.11.2025',
 '12.11.2025',
 '13.11.2025',
 '14.11.2025',
 '17.11.2025',
 '19.11.2025',
 '20.11.2025',
 '21.11.2025',
 '24.11.2025',
 '25.11.2025',
 '26.11.2025',
 '28.11.2025',
 'All']

In [168]:
df_bnp

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,03.11.2025,PRZELEW OTRZYMANY ELIXIR,DANIEL LILLA,33-100 TARNÓW,31102049550000750202263465,YOLO,CEN2511031646315,"1 500,00 1 513,17",NaN,NaN,NaN,NaN,NaN,NaN
0,03.11.2025,PRZELEW OTRZYMANY ELIXIR,DANIEL LILLA,33-100 TARNÓW,31102049550000750202263465,YOLO,CEN2511031646315,"1 500,00 1 513,17",NaN,NaN,NaN,NaN,NaN,NaN
0,04.11.2025,TRANS.BEZGOT. KARTĄ DEBET.,516842------5081,DANIEL LILLA Prague,HBO CZE,"29,99 PLN 2025-11-03",PSD3299895445467,"-29,99 1 483,18",NaN,NaN,NaN,NaN,NaN,NaN
0,03.11.2025,PRZELEW OTRZYMANY ELIXIR,DANIEL LILLA,33-100 TARNÓW,31102049550000750202263465,YOLO,CEN2511031646315,"1 500,00 1 513,17",NaN,NaN,NaN,NaN,NaN,NaN
0,04.11.2025,TRANS.BEZGOT. KARTĄ DEBET.,516842------5081,DANIEL LILLA Prague,HBO CZE,"29,99 PLN 2025-11-03",PSD3299895445467,"-29,99 1 483,18",NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,25.11.2025,TRANS.BEZGOT. KARTĄ DEBET.,516842------5081,DANIEL LILLA KRAKOW,PIEKARNIA BUCZEK POL,"4,40 PLN 2025-11-24",PSD6400990502513,"-4,40 189,52",NaN,NaN,NaN,NaN,NaN,NaN
0,26.11.2025,OPŁATA ZA RACHUNEK/PAKIET,OPŽATA ZA RACHUNEK/PAKIET,8324-378RPG-011,"-2,50 187,02",0,0,0,0,"-4,40 189,52",NaN,NaN,NaN,NaN
0,"-2,50 187,02",26.11.2025,TRANS.BEZGOT. KARTĄ DEBET.,DANIEL LILLA Krakow,Action S.A. A265 POL,"17,90 PLN 2025-11-25",PSD3238988617660,"-17,90 169,12",NaN,NaN,NaN,NaN,NaN,NaN
0,26.11.2025,BLIK - transakcja w POS,"Transakcja BLIK, Płatność BLIK w te","rminalu płatniczym, Nr 92178743826,",JMP S.A. BIEDRONKA,CEN2511260855619,"-42,92 126,20",NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [79]:
a[60:90]

['516842------5081',
 'DANIEL LILLA         KRAKOW',
 'PIEKARNIA BUCZEK          POL',
 '4,40              PLN 2025-11-24',
 'PSD6400990502513',
 '-4,40 189,52',
 '26.11.2025',
 'OPŁATA ZA RACHUNEK/PAKIET',
 'OPŽATA ZA RACHUNEK/PAKIET',
 '8324-378RPG-011',
 '-2,50 187,02',
 '26.11.2025',
 'TRANS.BEZGOT. KARTĄ DEBET.',
 '516842------5081',
 'DANIEL LILLA         Krakow',
 'Action S.A. A265          POL',
 '17,90             PLN 2025-11-25',
 'PSD3238988617660',
 '-17,90 169,12',
 '26.11.2025',
 'BLIK - transakcja w POS',
 'JMP S.A. BIEDRONKA',
 'SOLTYSOWSKA .',
 'KRAKOW',
 'PL',
 'Transakcja BLIK, Płatność BLIK w te',
 'rminalu płatniczym, Nr 92178743826,',
 'JMP S.A. BIEDRONKA',
 'CEN2511260855619',
 '-42,92 126,20']